# C4 · Reproject, resample and mosaic: the DEM on a grid in metres

> **Time:** about 2 hours  
> **You need:** Parts A and B (especially B4 · Coordinate reference systems in practice), and Lessons C1 to C3: the transform and `rowcol` (C1), maps with `extent=` (C2), and rioxarray's `open_rasterio`, `clip_box` and `to_raster` (C2, C3).  
> **By the end you can:**
> 1. explain why a raster must be **resampled** when it is reprojected, and reproject the DEM to UTM 45N with `.rio.reproject`, choosing the cell size,
> 2. choose a **resampling method**: nearest for categories, bilinear or cubic for smooth values, average for coarser cells, and line up two rasters with `reproject_match`,
> 3. split a raster into tiles and **mosaic** them back together, with nodata declared first, and prove the result equals the original.

In Lesson B4 you moved vector layers between coordinate systems with `to_crs`: every vertex got new coordinates, and the shapes came along. A raster cannot do that. Its cells are squares in a fixed grid, and in a new CRS they no longer fit any grid of squares. Reprojecting a raster means building a **new grid** and filling every new cell with a value taken from the old one.

This lesson puts the DEM on the grid that the rest of Part C uses: **UTM zone 45N, 30 m cells**. Slope (C6) and areas (C5) need metres, and that grid has them.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).* The full licence notice is in `data/README.md`.

**Counting lines:** lines that only set up or show a plot (`fig, ax = plt.subplots()`, `plt.show()`) do not count toward the Q1/Q2/Q3 line counts.

**About the numbers:** resampling can differ very slightly between library versions, so your decimals may differ a little from the text.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 28 of 39 · [Course home](../README.md)  
← Previous: [C3 · Clip, mask and subset: cutting the DEM to the Kathmandu Valley and its districts](C03_Clip_mask_and_subset.ipynb) · Next: [C5 · Raster algebra and reclassification: elevation zones of the Kathmandu Valley](C05_Raster_algebra_and_reclassification.ipynb) →

## 1. Why a raster must be resampled

The DEM's cells are squares **in degrees**: 1 arc-second each way. On the ground they are about 27.3 m wide and 30.8 m tall, as you worked out in C1.

In UTM 45N (B4) each of those cells becomes a slightly **tilted** shape, not quite a rectangle. A new grid of 30 m squares in UTM cannot follow those shapes. Each new square overlaps parts of several old cells.

So reprojection happens in two parts:

1. Build a **new grid**: a CRS, a transform and a shape (rows and columns).
2. Fill each new cell with a value from the old grid. This is **resampling**.

Vectors never need step 2, because a vertex is a point, not an area.

The example opens the DEM with rioxarray, as in C2 and C3: `masked=True` turns nodata into `NaN`, and `.squeeze("band", drop=True)` removes the band dimension of a one-band raster. Then it turns the 3 × 3 cells around Kathmandu Durbar Square into polygons, converts them to UTM 45N, and draws them on a grid of 30 m squares.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
import rioxarray as rxr
from rasterio.enums import Resampling
from rasterio.transform import rowcol
from shapely.geometry import box

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)
DEM_PATH = DATA / "dem_kathmandu_melamchi.tif"

dem = rxr.open_rasterio(DEM_PATH, masked=True).squeeze("band", drop=True)
print(dem.rio.crs, dem.shape, dem.dtype)
print("cell size (degrees):", dem.rio.resolution())

In [ ]:
dem_transform = dem.rio.transform()
row0, col0 = rowcol(dem_transform, 85.307, 27.704)     # the cell of Durbar Square (C1)

cells = []
for row in range(row0, row0 + 3):
    for col in range(col0, col0 + 3):
        west, north = dem_transform @ (col, row)          # top-left corner (C1)
        east, south = dem_transform @ (col + 1, row + 1)  # bottom-right corner
        cells.append(box(west, south, east, north))

cells_utm = gpd.GeoSeries(cells, crs="EPSG:4326").to_crs(32645)   # to_crs, as in B4
print("area of one cell:", round(cells_utm.area.iloc[0], 1), "m²")

fig, ax = plt.subplots(figsize=(6, 6))   # fig: the whole figure; ax: the plot area inside it (A10)
xmin, ymin, xmax, ymax = cells_utm.total_bounds
for x in np.arange(xmin // 30 * 30, xmax + 30, 30):      # grey lines every 30 m
    ax.axvline(x, color="grey", linewidth=0.8)
for y in np.arange(ymin // 30 * 30, ymax + 30, 30):
    ax.axhline(y, color="grey", linewidth=0.8)
cells_utm.boundary.plot(ax=ax, color="crimson", linewidth=2)
ax.set_xlim(xmin - 20, xmax + 20)
ax.set_ylim(ymin - 20, ymax + 20)
ax.ticklabel_format(style="plain", useOffset=False)   # full numbers on the axes
ax.tick_params(axis="x", labelrotation=45)
ax.set_title("Nine DEM cells (red) on a grid of 30 m squares (grey)")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

The DEM is in EPSG:4326, with 1,620 × 2,160 cells of 0.000277…° (one arc-second). With `masked=True` its dtype is `float32`, because `NaN` is a float (B6, C2).

In UTM each old cell covers about 843.2 m², a little less than the 900 m² of a 30 m square. The red cells are taller than they are wide, and their edges lean slightly against the grey grid. No grey square matches a red cell, so each new cell must **choose** or **blend** values from the old cells around it.

### Practice

**Q1 · one line:** How many rows and columns of cells did the example draw? Print `len(cells_utm)`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(len(cells_utm))
```

9: three rows of three cells. Each one is a polygon, so B4's `to_crs` could move it like any vector.
</details>

**Q2 · two lines:** Measure the red cells against the grey squares: divide the area of the first cell by 900 m², the area of one 30 m square, and print the ratio rounded to 3 decimals. Then print the total area of all nine cells in m², rounded to 0 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(cells_utm.area.iloc[0] / 900, 3))
print(round(cells_utm.area.sum(), 0))
```

A ratio of about 0.937: one old cell is about 6% smaller than one new square. The nine cells cover about 7,589 m². Area is meaningful here because `cells_utm` is in metres (B4).
</details>

**Q3 · three lines:** Is the tilt real or a drawing effect? Take the corner points of the first red cell with `list(cells_utm.iloc[0].exterior.coords)` and print the first two. The first two corners lie on the cell's right edge. Compare their eastings.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
corners = list(cells_utm.iloc[0].exterior.coords)
print(corners[0])
print(corners[1])
```

The two corners on the same (right) edge have eastings that differ by about 0.4 m over about 31 m of northing. In degrees that edge was exactly north–south; in UTM it leans. The DEM lies west of zone 45N's central meridian (87° E), where grid north and true north differ slightly.
</details>

## 2. Reprojecting with `.rio.reproject`

rioxarray reprojects a raster in one line: `dem.rio.reproject("EPSG:32645")`. It builds the new grid and resamples into it.

Which UTM zone? Each zone is 6° of longitude wide, numbered from 180° W. The zone number is `int((lon + 180) / 6) + 1`. All of the DEM, 85.2° to 85.8° E, lies in **zone 45** (84° to 90° E). Choosing the wrong zone is a common mistake; review 1 shows what it does.

Without more instructions, rioxarray chooses a cell size that keeps about the same number of cells, and makes the cells **square**. The new grid is a rectangle in UTM that **encloses** the tilted DEM, so its corners hold no data: they become `NaN`.

**Predict:** will the cell size in metres be a round number?

In [ ]:
dem_utm_default = dem.rio.reproject("EPSG:32645")

print(dem_utm_default.rio.crs, dem_utm_default.shape)
print("cell size (m):", dem_utm_default.rio.resolution())
print("cells with no data:", int(dem_utm_default.isnull().sum()))

In [ ]:
left, bottom, right, top = dem_utm_default.rio.bounds()
extent_km = [left / 1000, right / 1000, bottom / 1000, top / 1000]   # metres to km, for readable axes

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(dem_utm_default.values, extent=extent_km, cmap="terrain")
fig.colorbar(im, ax=ax, label="Elevation (m)", shrink=0.8)
ax.set_title("The DEM in UTM 45N (EPSG:32645)")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
plt.show()

The new grid has 1,766 rows and 2,085 columns, and its cells are about **28.65 m** square: not a round number. The `-28.65…` for y is negative, like `e` in the transform (C1): rows run south.

89,334 cells hold `NaN`. They form the thin white wedges along the edges of the map: the DEM's footprint is slightly tilted inside the new rectangle. Matplotlib leaves `NaN` cells blank.

The CRS is in metres, so the cells are square on the map with no aspect correction (compare C2). `rio.bounds()` gives the new grid's edges in metres; dividing them by 1,000 labels the axes in km, which is quicker to read than 3,090,000.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).*

### Practice

**Q1 · one line:** How many cells does `dem_utm_default` hold in total? Use `.size`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
dem_utm_default.size
```

3,682,110 cells, against 3,499,200 in the original. The new rectangle must enclose the tilted footprint, so it adds empty corner cells.
</details>

**Q2 · two lines:** Use the zone formula to print the UTM zone of the DEM's western edge (85.2° E) and of its eastern edge (85.8° E).

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(int((85.2 + 180) / 6) + 1)
print(int((85.8 + 180) / 6) + 1)
```

Both give zone 45, so one zone covers the whole DEM. For all of Nepal, which crosses 84° E, the west would fall in zone 44 (B4).
</details>

**Q3 · three lines:** How big is the new grid on the ground? Using `left`, `bottom`, `right` and `top` from the example, print its width and height in km, each rounded to 1 decimal place.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
width_km = (right - left) / 1000
height_km = (top - bottom) / 1000
print(round(width_km, 1), round(height_km, 1))
```

About 59.7 km × 50.6 km. In C1 you estimated about 58.9 km × 50.0 km for the DEM itself. The UTM rectangle is a little larger, because it encloses the tilted footprint and its empty corners.
</details>

## 3. Choosing the cell size and the resampling method

Give `resolution=30` and rioxarray builds a grid of exact **30 m** cells. Round numbers make every later step simpler: one cell is 30 × 30 = 900 m², or 0.0009 km².

The `resampling=` argument chooses how each new cell gets its value. The methods live in `rasterio.enums.Resampling`:

| Method | How a new cell gets its value | Good for |
| --- | --- | --- |
| `Resampling.nearest` | the value of the closest old cell | categories (codes), and the default |
| `Resampling.bilinear` | a weighted average of the 4 nearest old cells | smooth surfaces: elevation, rainfall |
| `Resampling.cubic` | a smooth curve through the 16 nearest old cells | smooth surfaces, sharper look |
| `Resampling.average` | the mean of all old cells inside the new one | making cells **larger** (step 5) |

Bilinear can never go beyond the values it averages. Cubic can **overshoot**: near a sharp change it may invent values a little lower or higher than any original cell.

**Predict:** which method will keep the DEM's lowest value, 525 m, exactly?

In [ ]:
methods = {"nearest": Resampling.nearest, "bilinear": Resampling.bilinear, "cubic": Resampling.cubic}

for name, method in methods.items():
    grid = dem.rio.reproject("EPSG:32645", resolution=30, resampling=method)
    print(f"{name:9} lowest {float(grid.min()):7.1f} m   highest {float(grid.max()):7.1f} m")

In [ ]:
# The grid used for the rest of Part C: UTM 45N, 30 m cells, bilinear
dem_utm = dem.rio.reproject("EPSG:32645", resolution=30, resampling=Resampling.bilinear)

print(dem_utm.shape, dem_utm.rio.resolution())
print(dem_utm.rio.transform())
print("cells with data:", int(dem_utm.notnull().sum()))

dem_utm.rio.to_raster(OUT / "c4_dem_utm30.tif")

**Nearest** keeps 525.0 m and 3,979.0 m exactly: it only copies old values. **Bilinear** gives about 525.3 m and 3,977.2 m: averaging pulls the extremes slightly inwards. **Cubic** gives about 524 m as its lowest value, **below** anything in the original DEM: an overshoot. For elevation, bilinear is the usual safe choice.

`dem_utm` has 1,687 rows and 1,991 columns of 30 m cells. Its transform starts at easting 322,259.37 m and northing 3,098,523.49 m, with `a = 30` and `e = −30`, as in C1. 3,276,043 cells hold data.

`to_raster` saved it as `c4_dem_utm30.tif`. rioxarray remembers how the original file stored its values (`int16`, nodata −32768) and writes the new file the same way, so the decimals from bilinear resampling are rounded to whole metres.

### Practice

**Q1 · one line:** What area does `dem_utm` cover with data? Multiply the number of cells with data by 0.0009 km² and round to 1 decimal place.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(int(dem_utm.notnull().sum()) * 0.0009, 1)
```

About 2,948.4 km². In C1 you estimated about 2,943 km² from degrees; counting 30 m cells in UTM is the better measurement.
</details>

**Q2 · two lines:** Open the saved file with rasterio and print its CRS, cell size, dtype and nodata value.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
with rasterio.open(OUT / "c4_dem_utm30.tif") as src:
    print(src.crs, src.res, src.dtypes, src.nodata)
```

EPSG:32645, (30.0, 30.0), `int16` and −32768. The file is in metres now, and stores whole metres like the original.
</details>

**Q3 · three lines:** Compare the means. Print the mean elevation of `dem` (degrees grid) and of `dem_utm` (UTM grid), both rounded to 1 decimal place, and their difference.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
mean_deg = float(dem.mean())
mean_utm = float(dem_utm.mean())
print(round(mean_deg, 1), round(mean_utm, 1), round(mean_deg - mean_utm, 1))
```

About 1,531.2 m and 1,531.0 m: a difference of about 0.2 m. xarray's `.mean()` skips the `NaN` corners. Resampling moves single cells by a few metres but barely changes the big picture.
</details>

## 4. Categories need `nearest`

Some rasters hold **codes**, not measurements: land-cover classes, district numbers, hazard zones. The numbers are labels. A code of 2.5 between "zone 2" and "zone 3" means nothing.

Bilinear and cubic **average** neighbouring values, so they invent such in-between codes along every boundary. For categories always use `Resampling.nearest`, which only copies existing codes.

The example makes a small category raster on purpose: `dem // 1000` gives 0 below 1,000 m, 1 from 1,000 to 1,999 m, and so on (whole-number division, A2). These are made-up teaching codes. Then it reprojects them both ways and lists the distinct values with `np.unique`.

In [ ]:
thousands = dem // 1000      # made-up category codes 0, 1, 2, 3
print(np.unique(thousands.values))

codes_nearest = thousands.rio.reproject("EPSG:32645", resolution=30, resampling=Resampling.nearest)
codes_bilinear = thousands.rio.reproject("EPSG:32645", resolution=30, resampling=Resampling.bilinear)

print("nearest: ", np.unique(codes_nearest.values))
print("bilinear:", len(np.unique(codes_bilinear.values)), "different values")
print(np.unique(codes_bilinear.values)[:5])

The codes are 0, 1, 2 and 3, plus `nan`, which `np.unique` lists once for all the empty cells.

With `nearest`, the reprojected grid holds the same four codes. With `bilinear` it holds tens of thousands of different values, such as 0.00002: averages of codes along the boundaries between zones, which are no code at all. A map coloured by code would look almost right, while every count and area per class would be wrong.

The rule: **continuous values** (elevation, rainfall) → bilinear or cubic; **categories** (codes) → nearest; making **categories coarser** → `Resampling.mode`, the most common code.

### Practice

**Q1 · one line:** Count how many cells of `codes_nearest` hold code 3 (3,000 m and above).

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
int((codes_nearest == 3).sum())
```

About 67,000 cells (the exact count can differ by a few between versions), which is about 60 km² at 0.0009 km² per cell.
</details>

**Q2 · two lines:** Which share of the cells of `codes_bilinear` holds a whole number? Compare the grid with `codes_bilinear.round()`, then divide the count of matches by the number of cells with data. Print it as a percentage, rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
whole = int((codes_bilinear == codes_bilinear.round()).sum())
print(round(whole / int(codes_bilinear.notnull().sum()) * 100, 1))
```

About 98% of the cells still hold a whole code: inside each zone, the four neighbours share one code. The broken codes sit along the boundaries, which is exactly where class areas are measured.
</details>

**Q3 · three lines:** Make the codes coarser in the right way: reproject `thousands` to 90 m cells with `Resampling.mode`, then print the distinct values and the new shape.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
codes_90 = thousands.rio.reproject("EPSG:32645", resolution=90, resampling=Resampling.mode)
print(np.unique(codes_90.values))
print(codes_90.shape)
```

Only the codes 0 to 3 (and `nan`) remain, on a grid of 563 × 664 cells. `mode` gives each large cell the code that is most common inside it, so no new codes appear.
</details>

## 5. Coarser cells: from 30 m to 90 m

Making cells **larger** is called **downsampling** or coarsening; making them smaller is **upsampling**. Upsampling never adds detail: it only spreads the old values over more cells.

Coarse grids are smaller and faster, and some data only exist at 90 m. But each large cell is an average, so **peaks get lower and valleys get higher**. For hazards this matters: slopes measured on a coarse DEM come out gentler than they are (C6).

Two ways to get 90 m cells:

- reproject with `resolution=90` and `Resampling.average`, which averages every 30 m cell that falls inside a 90 m cell,
- or take the 30 m grid and average blocks of 3 × 3 cells with `.coarsen(x=3, y=3, boundary="trim").mean()`. `boundary="trim"` drops the leftover rows and columns at the edge.

In [ ]:
dem_90 = dem.rio.reproject("EPSG:32645", resolution=90, resampling=Resampling.average)
print(dem_90.shape, dem_90.rio.resolution())

dem_90_blocks = dem_utm.coarsen(x=3, y=3, boundary="trim").mean()
print(dem_90_blocks.shape, "| x spacing:", float(dem_90_blocks.x[1] - dem_90_blocks.x[0]), "m")

print("highest, 30 m:", round(float(dem_utm.max()), 1), "m")
print("highest, 90 m:", round(float(dem_90.max()), 1), "m")

In [ ]:
# A 6 km x 6 km box in the hills north-east of the valley, at both cell sizes (clip_box, C3)
box_30 = dem_utm.rio.clip_box(350000, 3075000, 356000, 3081000)
box_90 = dem_90.rio.clip_box(350000, 3075000, 356000, 3081000)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, grid, label in zip(axes, [box_30, box_90], ["30 m cells", "90 m cells"]):
    l, b, r, t = grid.rio.bounds()
    im = ax.imshow(grid.values, extent=[l / 1000, r / 1000, b / 1000, t / 1000], cmap="terrain", vmin=800, vmax=2200)
    ax.set_title(f"{label}: {grid.shape[0]} x {grid.shape[1]}")
    ax.set_xlabel("Easting (km)")
axes[0].set_ylabel("Northing (km)")
fig.colorbar(im, ax=axes, label="Elevation (m)", shrink=0.8)
plt.show()

The 90 m grid has 563 × 664 cells, about one ninth of the 30 m grid. The `coarsen` result is one row and one column smaller (562 × 663), because `boundary="trim"` drops the edge cells that do not fill a complete 3 × 3 block; its x coordinates step by 90.0 m.

The highest cell drops from about 3,977.2 m to about 3,971.2 m. On the maps, the 90 m version shows the same ridges and valleys, but blocky: the small side valleys fade. Both maps share `vmin` and `vmax`, so equal colours mean equal elevations.

One caution: after `.coarsen`, `rio.resolution()` may still report the old 30 m until the grid is written to a file. The coordinates (90 m apart) are the truth.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).*

### Practice

**Q1 · one line:** How many times more cells does `dem_utm` hold than `dem_90`? Divide the two `.size` values and round to 2 decimals.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(dem_utm.size / dem_90.size, 2)
```

About 8.98, close to 9: three times fewer cells in each direction. The small difference comes from the edges, where the two grids do not end at the same place.
</details>

**Q2 · two lines:** Save `dem_90` as `c4_dem_utm90.tif` in `OUT`, then print the size of both saved files (30 m and 90 m) in MB, rounded to 2 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
dem_90.rio.to_raster(OUT / "c4_dem_utm90.tif")
print(round((OUT / "c4_dem_utm30.tif").stat().st_size / 1000000, 2), round((OUT / "c4_dem_utm90.tif").stat().st_size / 1000000, 2))
```

About 6.72 MB against 0.75 MB: nine times fewer cells, nine times less storage. These files are not compressed; review 2 shrinks one with `compress="deflate"`.
</details>

**Q3 · three lines:** Find the elevation at Kathmandu Durbar Square (85.307° E, 27.704° N) on both UTM grids. Make the point with `gpd.points_from_xy` in EPSG:4326 and convert it to UTM 45N (B4). Then use `rowcol` with each grid's transform, as in C1, and print both values rounded to 0 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
durbar = gpd.GeoSeries(gpd.points_from_xy([85.307], [27.704]), crs="EPSG:4326").to_crs(32645).iloc[0]
for grid in [dem_utm, dem_90]:
    print(round(float(grid.values[rowcol(grid.rio.transform(), durbar.x, durbar.y)]), 0))
```

About 1,301 m on the 30 m grid and about 1,302 m on the 90 m grid. On the flat valley floor coarsening barely matters; on a ridge the two could differ by tens of metres. The point must be in the grid's CRS before `rowcol`, or the row and column are meaningless.
</details>

## 6. Lining up two rasters with `reproject_match`

Two rasters can be combined cell by cell only when they are **aligned** (or **co-registered**): the same CRS, the same transform and the same shape. Then cell [row, col] means the same piece of ground in both.

`dem_utm` (30 m) and `dem_90` (90 m) share a CRS but not a grid. xarray does not stop you from subtracting them. It pairs cells whose coordinate **labels** match exactly and quietly drops the rest.

`other.rio.reproject_match(reference)` resamples `other` onto the reference grid: it copies the reference's CRS, transform and shape. You never type bounds or cell sizes by hand, so you cannot, for example, give latitude and longitude bounds where metres are needed.

**Predict:** what shape will `dem_utm - dem_90` have?

In [ ]:
print("silent mismatch:", (dem_utm - dem_90).shape)

dem_90_on_30 = dem_90.rio.reproject_match(dem_utm, resampling=Resampling.bilinear)
print(dem_90_on_30.shape, dem_90_on_30.rio.transform() == dem_utm.rio.transform())

diff_m = dem_utm - dem_90_on_30
print("mean absolute difference:", round(float(abs(diff_m).mean()), 1), "m")
print("range:", round(float(diff_m.min()), 1), "to", round(float(diff_m.max()), 1), "m")

In [ ]:
left, bottom, right, top = dem_utm.rio.bounds()
extent_km = [left / 1000, right / 1000, bottom / 1000, top / 1000]

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(diff_m.values, extent=extent_km, cmap="RdBu_r", vmin=-20, vmax=20)
fig.colorbar(im, ax=ax, label="30 m DEM minus 90 m DEM (m)", shrink=0.8)
ax.set_title("What the 90 m grid smooths away")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
plt.show()

The careless subtraction gave a grid of only 562 × 664 cells: xarray kept the cells whose x and y labels happened to match and dropped about 3 million others, without a warning.

After `reproject_match`, the 90 m grid has the 30 m grid's shape, 1,687 × 1,991, and the same transform (`True`). Now the subtraction is honest. The two grids differ by about 4.3 m on average, but by up to roughly 50 to 60 m in places.

On the map, red marks cells where the 30 m DEM is **higher** (ridges and spurs) and blue where it is **lower** (narrow valleys). The flat valley floor is white: there, coarsening changes almost nothing.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).*

### Practice

**Q1 · one line:** Print the shape of `dem_90` before matching, next to the shape of `dem_90_on_30` after it.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(dem_90.shape, dem_90_on_30.shape)
```

(563, 664) before and (1687, 1991) after. `reproject_match` made more cells, but no more detail: each new cell is interpolated from the 90 m values.
</details>

**Q2 · two lines:** What share of the cells with data differ by more than 10 m between the two grids? Count them with `abs(diff_m) > 10`, divide by the cells with data in `diff_m`, and print the percentage rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
big = int((abs(diff_m) > 10).sum())
print(round(big / int(diff_m.notnull().sum()) * 100, 1))
```

About 10% of the cells differ by more than 10 m, almost all on steep ground (the map's coloured lines). For a flat-land study 90 m may do; in the hills it hides real relief.
</details>

**Q3 · three lines:** Check alignment in one line of code per fact: print whether `dem_90_on_30` and `dem_utm` have the same CRS, the same shape and the same bounds.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(dem_90_on_30.rio.crs == dem_utm.rio.crs)
print(dem_90_on_30.shape == dem_utm.shape)
print(dem_90_on_30.rio.bounds() == dem_utm.rio.bounds())
```

`True` three times. Run checks like these before any cell-by-cell calculation with two rasters: it takes seconds and prevents silent mistakes.
</details>

## 7. Mosaicking tiles back together

Large DEMs and satellite images come as **tiles**: neighbouring files that share edges. Joining them into one raster is called a **mosaic**.

To practise, the example cuts the DEM into a western and an eastern half **on purpose**, saves each as a file, and joins them again.

- It opens the DEM **without** `masked=True`, so the values stay `int16` whole metres, as in the file.
- `.isel(x=slice(0, 1080))` selects columns 0 to 1079 by **position**, like `array[:, 0:1080]` in NumPy (A8).
- `rasterio.merge.merge(list_of_files)` reads the tiles and returns two things: an array of shape (bands, rows, columns) and the mosaic's transform.

The test of success: the mosaic must equal the original, value for value.

In [ ]:
from rasterio.merge import merge

dem_int = rxr.open_rasterio(DEM_PATH).squeeze("band", drop=True)   # int16, nodata -32768
west = dem_int.isel(x=slice(0, 1080))       # columns 0 to 1079
east = dem_int.isel(x=slice(1080, None))    # columns 1080 to the end
print(west.shape, east.shape)

west.rio.to_raster(OUT / "c4_west.tif")
east.rio.to_raster(OUT / "c4_east.tif")

mosaic, mosaic_transform = merge([OUT / "c4_west.tif", OUT / "c4_east.tif"])
print(mosaic.shape, mosaic.dtype)
print("same values as the original:", np.array_equal(mosaic[0], dem_int.values))
print("transform exactly equal:", mosaic_transform == dem_int.rio.transform())
print("transform equal within rounding:", mosaic_transform.almost_equals(dem_int.rio.transform()))

Each half has 1,620 rows and 1,080 columns. The mosaic has shape (1, 1620, 2160): one band, and the full grid again, in `int16`.

`np.array_equal` compares two arrays value by value: `True`, so not a single cell changed.

The transforms are **not** exactly equal (`False`), yet `almost_equals` says `True`. The corner coordinates differ only in the last digits of a float, a rounding effect of computing them twice. Compare floats with a tolerance, never with `==`.

### Practice

**Q1 · one line:** Which bounds does the western tile cover? Print `west.rio.bounds()`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(west.rio.bounds())
```

From about 85.2° to 85.5° E and from 27.55° to 28.0° N: the western half. Its right edge is exactly where the eastern tile begins.
</details>

**Q2 · two lines:** rioxarray can mosaic DataArrays in memory without files. Import `merge_arrays` from `rioxarray.merge`, join `[west, east]`, and print the shape and whether the values equal `dem_int`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
from rioxarray.merge import merge_arrays
joined = merge_arrays([west, east])
print(joined.shape, np.array_equal(joined.values, dem_int.values))
```

(1620, 2160) and `True`. `merge_arrays` returns a DataArray with its CRS and coordinates, ready for `.rio` methods. `merge` works on files, which suits downloaded tiles.
</details>

**Q3 · three lines:** Cut the DEM into a **northern** and a **southern** half instead (rows 0 to 809 and 810 to the end, with `.isel(y=...)`), join them with `merge_arrays`, and check the result.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
north = dem_int.isel(y=slice(0, 810))
south = dem_int.isel(y=slice(810, None))
print(np.array_equal(merge_arrays([north, south]).values, dem_int.values))
```

`True`. The tiles carry their own coordinates, so `merge_arrays` knows where each belongs, whichever way the DEM was cut.
</details>

## 8. Declare nodata first, then merge

Real tiles **overlap**, and they have **holes**: nodata cells at their edges or in gaps. Where tiles overlap, `merge` keeps the value from the **first** file in the list, unless that value is nodata. Then it takes the next file's value.

That only works if the files **declare** their nodata value. If a tile stores −32768 in a hole but does not say that −32768 means nodata, `merge` treats the hole as a real elevation of −32,768 m, and it wins over the good value of the next tile. This is a common mistake with real tiles.

The example builds the problem **on purpose**: two tiles with a 100-column overlap, and a 60 × 60 hole in the western tile, inside the overlap. Then `west_tile.rio.write_nodata(None)` makes the western file "forget" its nodata value.

In [ ]:
west_tile = dem_int.isel(x=slice(0, 1130)).copy()     # columns 0 to 1129
east_tile = dem_int.isel(x=slice(1030, None))          # columns 1030 to 2159: 100 columns overlap
west_tile.values[700:760, 1050:1110] = -32768          # a hole made ON PURPOSE, inside the overlap
west_tile = west_tile.rio.write_nodata(None)           # ... in a file that does not declare nodata

west_tile.rio.to_raster(OUT / "c4_west_hole.tif")
east_tile.rio.to_raster(OUT / "c4_east_overlap.tif")
tiles = [OUT / "c4_west_hole.tif", OUT / "c4_east_overlap.tif"]

with rasterio.open(tiles[0]) as src:
    print("nodata declared in the western file:", src.nodata)

bad, _ = merge(tiles)
print("hole cells left, nodata not declared:", int((bad == -32768).sum()))

good, _ = merge(tiles, nodata=-32768)
print("hole cells left, nodata=-32768:", int((good == -32768).sum()))
print("equal to the original:", np.array_equal(good[0], dem_int.values))

The western file declares no nodata (`None`). Merged as it is, all 3,600 hole cells (60 × 60) survive in the mosaic as −32,768 m, although the eastern tile had good values for every one of them.

With `nodata=-32768`, `merge` knows the hole is empty and fills it from the eastern tile: 0 hole cells, and the mosaic equals the original DEM.

`_` is a common Python name for a result you do not need (C3), here the transform. The lesson: **check and set nodata on every tile before merging**, then check the mosaic.

### Practice

**Q1 · one line:** Merge the same two files in the **opposite order**, eastern tile first, without `nodata=`. Count the −32768 cells.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
int((merge(tiles[::-1])[0] == -32768).sum())
```

0. Now the eastern tile comes first, so its good values win in the overlap. Order hides the problem here; it would not hide a hole outside the overlap. Declaring nodata is the real fix.
</details>

**Q2 · two lines:** Fix the tile itself: give `west_tile` its nodata value back with `.rio.write_nodata(-32768)`, save it over `c4_west_hole.tif`, and merge `tiles` again without `nodata=`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
west_tile.rio.write_nodata(-32768).rio.to_raster(OUT / "c4_west_hole.tif")
print(int((merge(tiles)[0] == -32768).sum()))
```

0 hole cells. `merge` takes the nodata value from the first file when you do not give one, so a file that declares its nodata is safe whatever the order.
</details>

**Q3 · three lines:** How badly did the bad mosaic damage the statistics? Print the mean of `bad[0]` and of `dem_int.values`, both rounded to 1 decimal, and the number of cells where they differ.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(bad[0].mean(), 1))
print(round(dem_int.values.mean(), 1))
print(int((bad[0] != dem_int.values).sum()))
```

About 1,496.0 m against 1,531.2 m: 3,600 wrong cells, about 0.1% of the grid, pulled the mean down by about 35 m. As in C1, one undeclared nodata value poisons every statistic.
</details>

## 9. When it breaks: subtracting grids that do not match

You want the difference between the 30 m and the 90 m DEMs, so you subtract their NumPy arrays. Run the cell.

In [ ]:
difference = dem_utm.values - dem_90.values

The last line of the error says:

`ValueError: operands could not be broadcast together with shapes (1687,1991) (563,664)`

- **ValueError** means an operation got values it cannot use.
- **operands** are the two things being subtracted.
- **broadcast** is NumPy's rule for combining arrays of different shapes (A8). (1687, 1991) and (563, 664) cannot be combined.

Here the error is your friend. NumPy refuses; xarray, in step 6, silently gave a 562 × 664 answer.

The fix: put both rasters on the same grid first, with `dem_90.rio.reproject_match(dem_utm)`, as in step 6. Then subtract.

**Try it:** fix the cell.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · The wrong UTM zone

Reproject `dem` to UTM zone **44N** (EPSG:32644) with 30 m cells and bilinear resampling. Print, for zone 44N and for zone 45N (`dem_utm`), the number of `NaN` cells and the area with data in km² (cells × 0.0009), rounded to 1 decimal.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
dem_44 = dem.rio.reproject("EPSG:32644", resolution=30, resampling=Resampling.bilinear)

for name, grid in [("44N", dem_44), ("45N", dem_utm)]:
    empty = int(grid.isnull().sum())
    area_km2 = int(grid.notnull().sum()) * 0.0009
    print(f"{name}: {empty:,} empty cells, {area_km2:.1f} km² with data")
```

Zone 44N gives 246,307 empty cells and about 2,961.2 km²; zone 45N gives 82,774 and about 2,948.4 km². Zone 44N is made for 78° to 84° E. The DEM lies east of it, so its grid is more tilted (three times more empty cells) and its distances are more stretched: the same ground measures about 13 km² more. No error warns you; only checking the zone does.
</details>

### Review 2 · A 90 m DEM, written and checked

Reproject the original DEM to UTM 45N with 90 m cells and `Resampling.average`. Save it as `c4_review_90m.tif` with `compress="deflate"` (lossless compression, as in the original DEM file of C1), read it back with rasterio, and print its CRS, cell size, shape, nodata value and file size in MB (2 decimals).

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
grid_90 = dem.rio.reproject("EPSG:32645", resolution=90, resampling=Resampling.average)
path_90 = OUT / "c4_review_90m.tif"
grid_90.rio.to_raster(path_90, compress="deflate")

with rasterio.open(path_90) as src:
    print(src.crs, src.res, src.shape, src.nodata)
print(round(path_90.stat().st_size / 1000000, 2), "MB")
```

EPSG:32645, (90.0, 90.0), (563, 664) and nodata −32768, in a file of about 0.55 MB (the exact size can differ slightly between versions). Reading the file back proves that the CRS and cell size were written, not only computed.
</details>

### Review 3 · Four tiles into one

Cut `dem_int` into four quadrants on purpose (north-west, north-east, south-west, south-east) at row 810 and column 1080. Join all four with `merge_arrays` and prove that the mosaic equals `dem_int`.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
from rioxarray.merge import merge_arrays

quadrants = []
for rows in [slice(0, 810), slice(810, None)]:
    for cols in [slice(0, 1080), slice(1080, None)]:
        quadrants.append(dem_int.isel(y=rows, x=cols))

print([q.shape for q in quadrants])
mosaic_4 = merge_arrays(quadrants)
print(mosaic_4.shape, np.array_equal(mosaic_4.values, dem_int.values))
```

Four tiles of 810 × 1080 cells join back into 1620 × 2160 cells, identical to the original (`True`). The nested loop (A3) walks through every pair of row and column ranges. A list comprehension inside `print` shows all four shapes at once.
</details>

### Review 4 · How far does coarsening move the peaks?

Put `dem_90` back on the 30 m grid with `reproject_match` (bilinear). Find the cell where the 30 m DEM is highest, with `np.nanargmax` and `np.unravel_index` (A8; `nanargmax` skips `NaN`), and print the elevation of that cell on both grids and the difference.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
back_on_30 = dem_90.rio.reproject_match(dem_utm, resampling=Resampling.bilinear)
values_30 = dem_utm.values
row, col = np.unravel_index(np.nanargmax(values_30), values_30.shape)

high_30 = float(values_30[row, col])
high_90 = float(back_on_30.values[row, col])
print(row, col)
print(round(high_30, 1), round(high_90, 1), round(high_30 - high_90, 1))
```

The highest 30 m cell (about 3,977 m) reads about 3,959.7 m on the coarsened grid: the summit lost about 17.5 m, because each 90 m cell averages its slopes with the top. On a summit or a ridge, a coarse DEM is always too low.
</details>

## Recap

- Reprojecting a raster builds a **new grid** (CRS, transform, shape) and **resamples** values into it, because old cells no longer fit the new squares. Vectors only move their vertices.
- `dem.rio.reproject("EPSG:32645", resolution=30, resampling=Resampling.bilinear)` gives the Part C grid: UTM 45N, 30 m cells, 0.0009 km² each. The DEM lies wholly in zone 45; check zones with `int((lon + 180) / 6) + 1`.
- **Nearest** for categories (it never invents codes), **bilinear** or **cubic** for smooth values (cubic can overshoot), **average** or **mode** for coarser cells. Coarse grids lower peaks and fill valleys.
- Cell-by-cell maths needs **aligned** rasters: same CRS, transform and shape. `reproject_match(reference)` aligns one raster to another; xarray may otherwise drop cells silently.
- Mosaic tiles with `rasterio.merge.merge` (files) or `merge_arrays` (DataArrays). **Declare nodata before merging**, then prove the mosaic with `np.array_equal`.

### Words you learned

| Word | Meaning |
| --- | --- |
| reprojection (raster) | Building a new grid in another CRS and filling it with values from the old grid |
| resampling | Choosing each new cell's value from the cells of another grid |
| nearest neighbour | Resampling that copies the value of the closest old cell |
| bilinear | Resampling that takes a distance-weighted average of the 4 nearest old cells |
| cubic | Resampling that fits a smooth curve through the 16 nearest old cells; it can overshoot |
| overshoot | A resampled value beyond the range of the original values |
| `Resampling.average` / `.mode` | The mean / the most common value of all old cells inside a larger new cell |
| downsampling / upsampling | Making cells larger (coarser) / smaller (finer) |
| `.coarsen` | xarray method that groups blocks of cells, for example 3 × 3, to make a coarser grid |
| aligned / co-registered | Rasters with the same CRS, transform and shape, so a cell means the same ground in both |
| `reproject_match` | rioxarray method that resamples a raster onto another raster's grid |
| tile | One of several neighbouring raster files that together cover a large area |
| mosaic | One raster made by joining tiles |
| `.isel` | xarray selection by position, like NumPy slicing |
| `almost_equals` | Compares two transforms allowing tiny rounding differences |

## Next

**C5 · Raster algebra and reclassification:** calculate with whole grids at once, avoid the integer overflow trap, sort the DEM into elevation zones and measure their areas in km².

<!-- pygile-nav -->
**PyGILE Course** · lesson 28 of 39 · [Course home](../README.md)  
← Previous: [C3 · Clip, mask and subset: cutting the DEM to the Kathmandu Valley and its districts](C03_Clip_mask_and_subset.ipynb) · Next: [C5 · Raster algebra and reclassification: elevation zones of the Kathmandu Valley](C05_Raster_algebra_and_reclassification.ipynb) →